In [493]:
states = ["Code", "Test", "Dependency", "CI/Config"] 

actions = [
    "Inspect pipeline logs",
    "Inspect failed pipeline stage",
    "Compare with last successful run",
    "Inspect recent code/test changes",
    "Inspect dependency changes",
    "Inspect CI/environment configuration",
    "Search previous incidents/runbooks",
    "Inspect Docker image/cache changes",
    "Inspect hosting environment",
    "Inspect server/session state",
    "Escalate to human"
]


priors = {
    "Code": 0.1770,
    "Test": 0.3934,
    "Dependency": 0.2984,
    "CI/Config": 0.1311
}

likelihoods = {
    "Build": {
        "Code": 0.574,
        "Test": 0.358,
        "Dependency": 0.429,
        "CI/Config": 0.275
    },
    "Test": {
        "Code": 0.185,
        "Test": 0.533,
        "Dependency": 0.132,
        "CI/Config": 0.100
    },
    "Setup / Dependency": {
        "Code": 0.074,
        "Test": 0.000,
        "Dependency": 0.176,
        "CI/Config": 0.125
    },
    "Quality / Analysis": {
        "Code": 0.019,
        "Test": 0.058,
        "Dependency": 0.055,
        "CI/Config": 0.050
    },
    "Deploy / Publish": {
        "Code": 0.000,
        "Test": 0.000,
        "Dependency": 0.033,
        "CI/Config": 0.050
    },
    "Ambiguous": {
        "Code": 0.074,
        "Test": 0.033,
        "Dependency": 0.077,
        "CI/Config": 0.150
    },
    "Unknown": {
        "Code": 0.074,
        "Test": 0.017,
        "Dependency": 0.099,
        "CI/Config": 0.250
    }
}

In [494]:
def calculate_posterior(failed_stage):
    stage_likelihoods = likelihoods[failed_stage]

    unnormalized = {}

    for state in states:
        unnormalized[state] = (
            priors[state] * stage_likelihoods[state]
        )

    evidence_probability = sum(unnormalized.values())

    posterior = {}

    for state in states:
        posterior[state] = (
            unnormalized[state] / evidence_probability
        )

    return posterior

In [495]:
posterior = calculate_posterior("Build")

for state, probability in posterior.items():
    print(f"{state}: {probability:.3%}")

Code: 24.993%
Test: 34.646%
Dependency: 31.492%
CI/Config: 8.869%


In [496]:
CONFIDENCE_THRESHOLD = 0.90


def should_stop(posterior):
    best_state = max(posterior, key=posterior.get)
    best_probability = posterior[best_state]

    if best_probability >= CONFIDENCE_THRESHOLD:
        return True, best_state

    return False, best_state

In [497]:
posterior = calculate_posterior("Build")

stop, best_state = should_stop(posterior)

print("Best hypothesis:", best_state)
print("Stop diagnosis:", stop)

Best hypothesis: Test
Stop diagnosis: False


In [498]:
actions = [
    {
        "name": "Inspect pipeline logs",
        "tags": ["all"],
        "cost": 1,
        "outcomes": [
            "Relevant error found",
            "No clear error found"
        ]
    },
    {
        "name": "Inspect failed pipeline stage",
        "tags": ["all"],
        "cost": 2,
        "outcomes": [
            "Specific failed stage identified",
            "Failure remains ambiguous"
        ]
    },
    {
        "name": "Compare with last successful run",
        "tags": ["all"],
        "cost": 2,
        "outcomes": [
            "Significant difference found",
            "No significant difference found"
        ]
    },
    {
        "name": "Inspect recent code/test changes",
        "tags": ["Build", "Test", "Quality / Analysis"],
        "cost": 1,
        "outcomes": [
            "Relevant code/test change found",
            "No relevant change found"
        ]
    },
    {
        "name": "Inspect dependency changes",
        "tags": ["Build", "Setup / Dependency"],
        "cost": 1,
        "outcomes": [
            "Dependency change found",
            "No dependency change found"
        ]
    },
    {
        "name": "Inspect CI/environment configuration",
        "tags": [
            "Build",
            "Test",
            "Setup / Dependency",
            "Quality / Analysis",
            "Deploy / Publish"
        ],
        "cost": 2.5,
        "outcomes": [
            "Configuration/environment issue found",
            "No configuration/environment issue found"
        ]
    },
    {
        "name": "Search previous incidents/runbooks",
        "tags": ["all"],
        "cost": 1,
        "outcomes": [
            "Similar incident found",
            "No similar incident found"
        ]
    },
    {
        "name": "Inspect Docker image/cache changes",
        "tags": ["Build", "Deploy / Publish"],
        "cost": 1.5,
        "outcomes": [
            "Docker/image difference found",
            "No Docker/image difference found"
        ]
    },
    {
        "name": "Inspect hosting environment",
        "tags": ["Deploy / Publish"],
        "cost": 2,
        "outcomes": [
            "Infrastructure issue found",
            "No infrastructure issue found"
        ]
    },
    {
        "name": "Inspect server/session state",
        "tags": ["Build", "Deploy / Publish"],
        "cost": 1.5,
        "outcomes": [
            "Server/session issue found",
            "No server/session issue found"
        ]
    }
]

import math

def entropy(probabilities):
    return -sum(
        p * math.log2(p)
        for p in probabilities
        if p > 0
    )


def calculate_eig(current_posterior, outcome_model):
    current_entropy = entropy(current_posterior.values())

    eig = 0

    for outcome, probabilities in outcome_model.items():

        # P(outcome | E, action)
        outcome_probability = sum(
            current_posterior[state] * probabilities[state]
            for state in states
        )

        # P(H | E, outcome, action)
        outcome_posterior = {}

        for state in states:
            numerator = (
                current_posterior[state]
                * probabilities[state]
            )

            outcome_posterior[state] = (
                numerator / outcome_probability
            )

        # Information gained from this outcome
        outcome_entropy = entropy(outcome_posterior.values())

        information_gain = (
            current_entropy - outcome_entropy
        )

        # Expected contribution of this outcome
        eig += outcome_probability * information_gain

    return eig

def rank_actions(posterior, candidate_actions, outcome_models):
    eig_known = []
    eig_unknown = []

    for action in candidate_actions:
        name = action["name"]
        cost = action["cost"]

        if name in outcome_models:
            eig = calculate_eig(
                posterior,
                outcome_models[name]
            )

            score = eig / cost

            eig_known.append({
                "name": name,
                "cost": cost,
                "eig": eig,
                "score": score
            })

        else:
            eig_unknown.append({
                "name": name,
                "cost": cost,
                "eig": None,
                "score": None
            })

    # EIG-known: highest information per cost first
    eig_known.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    # EIG-unknown: cheapest first
    eig_unknown.sort(
        key=lambda x: x["cost"]
    )

    return eig_known, eig_unknown


def rank_actions_by_cost(candidate_actions):
    ranked_actions = []

    for action in candidate_actions:
        ranked_actions.append({
            "name": action["name"],
            "cost": action["cost"]
        })

    ranked_actions.sort(
        key=lambda x: x["cost"]
    )

    return ranked_actions


def get_candidate_actions(failed_stage, actions):
    candidate_actions = []

    for action in actions:
        if "all" in action["tags"] or failed_stage in action["tags"]:
            candidate_actions.append(action)

    return candidate_actions


def decide_next_step(
    posterior,
    candidate_actions,
    outcome_models
):
    # 1. Check stopping criterion
    stop, best_state = should_stop(posterior)

    if stop:
        return {
            "decision": "REPORT",
            "root_cause": best_state
        }

    # 2. Rank remaining actions
    eig_known, eig_unknown = rank_actions(
        posterior,
        candidate_actions,
        outcome_models
    )

    # 3. Prefer EIG-known actions
    if eig_known:
        return {
            "decision": "RECOMMEND",
            "action": eig_known[0]
        }

    # 4. Fall back to unknown-EIG actions
    if eig_unknown:
        return {
            "decision": "RECOMMEND",
            "action": eig_unknown[0]
        }

    # 5. No actions remain and confidence is insufficient
    return {
        "decision": "ESCALATE"
    }

def decide_next_step_cost_only(
    posterior,
    candidate_actions
):
    # 1. Check stopping criterion
    stop, best_state = should_stop(posterior)

    if stop:
        return {
            "decision": "REPORT",
            "root_cause": best_state
        }

    # 2. No actions left
    if not candidate_actions:
        return {
            "decision": "ESCALATE"
        }

    # 3. Choose cheapest action
    ranked_actions = rank_actions_by_cost(
        candidate_actions
    )

    return {
        "decision": "RECOMMEND",
        "action": ranked_actions[0]
    }

def update_posterior_with_outcome(
    current_posterior,
    outcome_probabilities
):
    # P(o | E, a)
    outcome_probability = sum(
        current_posterior[state] * outcome_probabilities[state]
        for state in states
    )

    new_posterior = {}

    for state in states:
        numerator = (
            current_posterior[state]
            * outcome_probabilities[state]
        )

        new_posterior[state] = (
            numerator / outcome_probability
        )

    return new_posterior

def run_diagnosis(failed_stage, outcomes):
    posterior = calculate_posterior(failed_stage)

    remaining_actions = get_candidate_actions(
        failed_stage,
        actions
    ).copy()

    for outcome in outcomes:

        decision = decide_next_step(
            posterior,
            remaining_actions,
            outcome_models
        )

        print("\nDecision:", decision)

        if decision["decision"] == "REPORT":
            print("Root cause:", decision["root_cause"])
            return decision

        if decision["decision"] == "ESCALATE":
            print("Escalating to human.")
            return decision

        action = decision["action"]
        action_name = action["name"]

        print("Performing:", action_name)

        # Remove completed action
        remaining_actions = [
            a for a in remaining_actions
            if a["name"] != action_name
        ]

        # Simulated outcome
        outcome_probabilities = (
            outcome_models[action_name][outcome]
        )

        posterior = update_posterior_with_outcome(
            posterior,
            outcome_probabilities
        )

        print("Updated posterior:")
        for state, probability in posterior.items():
            print(f"{state}: {probability:.3%}")

    return posterior

In [499]:
outcome_models = {

    "Inspect pipeline logs": {
        "Relevant failure signal found": {
            "Code": 0.174,
            "Test": 0.036,
            "Dependency": 0.024,
            "CI/Config": 0.056
        },
        "No relevant failure signal found": {
            "Code": 0.826,
            "Test": 0.964,
            "Dependency": 0.976,
            "CI/Config": 0.944
        }
    },

    "Inspect failed pipeline stage": {
        "Stage-specific clue found": {
            "Code": 0.935,
            "Test": 0.938,
            "Dependency": 0.855,
            "CI/Config": 0.611
        },
        "No additional clue": {
            "Code": 0.065,
            "Test": 0.063,
            "Dependency": 0.145,
            "CI/Config": 0.389
        }
    },

    "Compare with last successful run": {
        "Meaningful difference found": {
            "Code": 0.022,
            "Test": 0.009,
            "Dependency": 0.012,
            "CI/Config": 0.056
        },
        "No meaningful difference": {
            "Code": 0.978,
            "Test": 0.991,
            "Dependency": 0.988,
            "CI/Config": 0.944
        }
    },

    "Inspect recent code/test changes": {
        "Relevant change found": {
            "Code": 0.978,
            "Test": 0.991,
            "Dependency": 0.012,
            "CI/Config": 0.028
        },
        "No relevant change found": {
            "Code": 0.022,
            "Test": 0.009,
            "Dependency": 0.988,
            "CI/Config": 0.972
        }
    },

    "Inspect dependency changes": {
        "Dependency-related change found": {
            "Code": 0.022,
            "Test": 0.009,
            "Dependency": 0.988,
            "CI/Config": 0.028
        },
        "No dependency-related change": {
            "Code": 0.978,
            "Test": 0.991,
            "Dependency": 0.012,
            "CI/Config": 0.972
        }
    },

    "Inspect CI/environment configuration": {
        "Configuration/environment issue found": {
            "Code": 0.022,
            "Test": 0.009,
            "Dependency": 0.169,
            "CI/Config": 0.972
        },
        "No configuration/environment issue": {
            "Code": 0.978,
            "Test": 0.991,
            "Dependency": 0.831,
            "CI/Config": 0.028
        }
    },

    "Search previous incidents/runbooks": {
        "Similar incident found": {
            "Code": 0.717,
            "Test": 0.938,
            "Dependency": 0.506,
            "CI/Config": 0.194
        },
        "No similar incident found": {
            "Code": 0.283,
            "Test": 0.063,
            "Dependency": 0.494,
            "CI/Config": 0.806
        }
    },

    "Inspect Docker image/cache changes": {
        "Image/cache issue found": {
            "Code": 0.022,
            "Test": 0.018,
            "Dependency": 0.048,
            "CI/Config": 0.111
        },
        "No image/cache issue": {
            "Code": 0.978,
            "Test": 0.982,
            "Dependency": 0.952,
            "CI/Config": 0.889
        }
    },

    "Inspect hosting environment": {
        "Environment issue found": {
            "Code": 0.022,
            "Test": 0.018,
            "Dependency": 0.169,
            "CI/Config": 0.111
        },
        "No environment issue": {
            "Code": 0.978,
            "Test": 0.982,
            "Dependency": 0.831,
            "CI/Config": 0.889
        }
    },

    "Inspect server/session state": {
        "Server/session issue found": {
            "Code": 0.022,
            "Test": 0.009,
            "Dependency": 0.012,
            "CI/Config": 0.056
        },
        "No server/session issue": {
            "Code": 0.978,
            "Test": 0.991,
            "Dependency": 0.988,
            "CI/Config": 0.944
        }
    }
}

In [500]:
failed_stage = "Build"

def test_v1():
    # --------------------------------------------------
    # 1. Test Bayesian posterior
    # --------------------------------------------------
    posterior = calculate_posterior("Build")

    assert abs(sum(posterior.values()) - 1.0) < 1e-9
    print("✓ Posterior calculation")


    # --------------------------------------------------
    # 2. Test posterior update
    # --------------------------------------------------
    new_posterior = update_posterior_with_outcome(
        posterior,
        outcome_models["Inspect recent code/test changes"]
            ["Relevant change found"]
    )

    assert abs(sum(new_posterior.values()) - 1.0) < 1e-9
    assert new_posterior["Code"] > posterior["Code"]

    print("✓ Posterior update")


    # --------------------------------------------------
    # 3. Test EIG
    # --------------------------------------------------
    eig = calculate_eig(
        posterior,
        outcome_models["Inspect recent code/test changes"]
    )

    assert eig >= 0
    print("✓ EIG calculation")


    # --------------------------------------------------
    # 4. Test tag filtering
    # --------------------------------------------------
    candidate_actions = get_candidate_actions(
        "Build",
        actions
    )

    for action in candidate_actions:
        assert (
            "all" in action["tags"]
            or "Build" in action["tags"]
        )

    print("✓ Action filtering")


    # --------------------------------------------------
    # 5. Test action ranking
    # --------------------------------------------------
    eig_known, eig_unknown = rank_actions(
        posterior,
        candidate_actions,
        outcome_models
    )

    assert any(
        action["name"] == "Inspect recent code/test changes"
        for action in eig_known
    )

    for action in eig_unknown:
        assert action["eig"] is None

    print("✓ Action ranking")


    # --------------------------------------------------
    # 6. Test recommendation
    # --------------------------------------------------
    decision = decide_next_step(
        posterior,
        candidate_actions,
        outcome_models
    )

    assert decision["decision"] == "RECOMMEND"

    print("✓ Recommendation")


    # --------------------------------------------------
    # 7. Test reporting
    # --------------------------------------------------
    high_confidence = {
        "Code": 0.95,
        "Test": 0.02,
        "Dependency": 0.02,
        "CI/Config": 0.01
    }

    decision = decide_next_step(
        high_confidence,
        candidate_actions,
        outcome_models
    )

    assert decision["decision"] == "REPORT"
    assert decision["root_cause"] == "Code"

    print("✓ Reporting")


    # --------------------------------------------------
    # 8. Test escalation
    # --------------------------------------------------
    low_confidence = {
        "Code": 0.40,
        "Test": 0.30,
        "Dependency": 0.20,
        "CI/Config": 0.10
    }

    decision = decide_next_step(
        low_confidence,
        [],
        outcome_models
    )

    assert decision["decision"] == "ESCALATE"

    print("✓ Escalation")


    print("\nAll V1 tests passed!")

In [501]:
test_v1()

✓ Posterior calculation
✓ Posterior update
✓ EIG calculation
✓ Action filtering
✓ Action ranking
✓ Recommendation
✓ Reporting
✓ Escalation

All V1 tests passed!


In [502]:
# Preparing test cases to evaluate our model

In [503]:
import pandas as pd

# Load the dataset
df = pd.read_excel("../data/replication-labeling.xlsx")


# --------------------------------------------------
# 1. Map dataset labels → our V1 hidden states
# --------------------------------------------------

def map_hidden_state(row):
    sub = str(row["sub-category"]).strip()

    if sub == "Software tests fail":
        return "Test"

    if sub == "Project compilation fails due to issues in the source code":
        return "Code"

    if sub in [
        "Project compilation fails due to dependency conflicts",
        "Project compilation fails due to unresolved dependencies",
        "Workflow environment setup fails due to dependency issue"
    ]:
        return "Dependency"

    if sub in [
        "Workflow execution fails due to workflow configuration issues or incorrect instructions",
        "Project compilation fails due to project configuration mistakes/issues",
        "Access to third-party services fails due to permission issue",
        "Workflow build fails due to hardware/resource limitation"
    ]:
        return "CI/Config"

    return None


df["true_root_cause"] = df.apply(map_hidden_state, axis=1)


# --------------------------------------------------
# 2. Remove cases that don't belong to our V1 model
# --------------------------------------------------

evaluation_pool = df[
    df["true_root_cause"].notna()
].copy()


# --------------------------------------------------
# 3. Convert the observed failed step into our
#    failed_stage evidence
# --------------------------------------------------

def normalize_stage(step):
    s = str(step).lower().strip()

    if any(
        keyword in s
        for keyword in ["test", "tests", "junit", "pytest", "cucumber"]
    ):
        return "Test"

    return "Build"


evaluation_pool["failed_stage"] = (
    evaluation_pool["step"]
    .apply(normalize_stage)
)


# --------------------------------------------------
# 4. Select 10 cases from each hidden state
# --------------------------------------------------

evaluation_cases = (
    evaluation_pool
    .groupby("true_root_cause", group_keys=False)
    .sample(n=10, random_state=42)
    .reset_index(drop=True)
)


# --------------------------------------------------
# 5. Give every case an evaluation ID
# --------------------------------------------------

evaluation_cases.insert(
    0,
    "case_id",
    range(1, len(evaluation_cases) + 1)
)


print("Evaluation cases:", len(evaluation_cases))

print(
    evaluation_cases["true_root_cause"]
    .value_counts()
)


# Show the cases we'll actually use
evaluation_cases[
    [
        "case_id",
        "failed_stage",
        "true_root_cause",
        "step",
        "root cause"
    ]
]

Evaluation cases: 40
true_root_cause
CI/Config     10
Code          10
Dependency    10
Test          10
Name: count, dtype: int64


,case_id,failed_stage,true_root_cause,step,root cause
0,1,Build,CI/Config,Run bytedeco/javacpp-presets/.Github/actions/d...,Maven build failure due to hardware/resource l...
1,2,Build,CI/Config,Build with Ant and ivy,Ant build failure due to Configuration Issue(p...
2,3,Build,CI/Config,Run actions/setup-java@v3,Configuration Issue due to unknown package man...
3,4,Test,CI/Config,Run integration test,Maven build failure due to startup of Docker
4,5,Build,CI/Config,Scan for dependencies and licenses,Dependency and license scan failure due to inv...
5,6,Build,CI/Config,Build dex-tools with Gradle,Gradle build failure due to Configuration Issu...
6,7,Build,CI/Config,Build and sonar analyze,Maven execution failure due to Configuration I...
7,8,Build,CI/Config,Update Dependency graph,Resource not accessible due to workflow permis...
8,9,Build,CI/Config,8. Upload coverage report,Codecov upload failure: Unable to locate build...
9,10,Test,CI/Config,Test Report,Maven build failure due to test report not acc...


In [504]:
import random


def simulate_action_outcome(
    action_name,
    true_root_cause,
    case_id
):
    outcomes = outcome_models[action_name]

    outcome_names = list(outcomes.keys())

    probabilities = [
        outcomes[outcome][true_root_cause]
        for outcome in outcome_names
    ]

    # Same case + same action = same outcome
    seed = hash(
        (case_id, action_name, true_root_cause)
    )

    rng = random.Random(seed)

    return rng.choices(
        outcome_names,
        weights=probabilities,
        k=1
    )[0]


def evaluate_case(case):
    """
    Run the diagnostic agent on one evaluation case.
    """


    failed_stage = case["failed_stage"]
    true_root_cause = case["true_root_cause"]

    posterior = calculate_posterior(failed_stage)

    remaining_actions = get_candidate_actions(
        failed_stage,
        actions
    ).copy()

    trajectory = []

    while True:

        decision = decide_next_step(
            posterior,
            remaining_actions,
            outcome_models
        )

        # ------------------------------------------
        # REPORT
        # ------------------------------------------
        if decision["decision"] == "REPORT":

            return {
                "case_id": case["case_id"],
                "true_root_cause": true_root_cause,
                "final_prediction": decision["root_cause"],
                "decision": "REPORT",
                "actions_taken": trajectory,
                "total_cost": sum(
                    step["cost"]
                    for step in trajectory
                ),
                "correct": (
                    decision["root_cause"] == true_root_cause
                )
            }

        # ------------------------------------------
        # ESCALATE
        # ------------------------------------------
        if decision["decision"] == "ESCALATE":

            return {
                "case_id": case["case_id"],
                "true_root_cause": true_root_cause,
                "final_prediction": None,
                "decision": "ESCALATE",
                "actions_taken": trajectory,
                "total_cost": sum(
                    step["cost"]
                    for step in trajectory
                ),
                "correct": False
            }

        # ------------------------------------------
        # RECOMMEND ACTION
        # ------------------------------------------

        action = decision["action"]
        action_name = action["name"]

        # Simulate what this diagnostic action discovers
        outcome = simulate_action_outcome(
            action_name,
            true_root_cause,
            case["case_id"]
        )

        # Save the action BEFORE updating belief
        trajectory.append({
            "action": action_name,
            "cost": action["cost"],
            "eig": action["eig"],
            "outcome": outcome,
            "posterior_before": posterior.copy()
        })

        # Remove completed action
        remaining_actions = [
            a for a in remaining_actions
            if a["name"] != action_name
        ]

        # Bayesian update using the observed outcome
        posterior = update_posterior_with_outcome(
            posterior,
            outcome_models[action_name][outcome]
        )

        trajectory[-1]["posterior_after"] = posterior.copy()

In [505]:
def evaluate_case_cost_only(case):

    

    failed_stage = case["failed_stage"]
    true_root_cause = case["true_root_cause"]

    posterior = calculate_posterior(failed_stage)

    remaining_actions = get_candidate_actions(
        failed_stage,
        actions
    ).copy()

    trajectory = []

    while True:

        decision = decide_next_step_cost_only(
            posterior,
            remaining_actions
        )

        # -----------------------------
        # REPORT
        # -----------------------------
        if decision["decision"] == "REPORT":

            return {
                "case_id": case["case_id"],
                "true_root_cause": true_root_cause,
                "final_prediction": decision["root_cause"],
                "decision": "REPORT",
                "actions_taken": trajectory,
                "total_cost": sum(
                    step["cost"]
                    for step in trajectory
                ),
                "correct": (
                    decision["root_cause"] == true_root_cause
                )
            }

        # -----------------------------
        # ESCALATE
        # -----------------------------
        if decision["decision"] == "ESCALATE":

            return {
                "case_id": case["case_id"],
                "true_root_cause": true_root_cause,
                "final_prediction": None,
                "decision": "ESCALATE",
                "actions_taken": trajectory,
                "total_cost": sum(
                    step["cost"]
                    for step in trajectory
                ),
                "correct": False
            }

        # -----------------------------
        # RECOMMEND ACTION
        # -----------------------------
        action = decision["action"]
        action_name = action["name"]

        outcome = simulate_action_outcome(
            action_name,
            true_root_cause,
            case["case_id"]
        )

        trajectory.append({
            "action": action_name,
            "cost": action["cost"],
            "outcome": outcome,
            "posterior_before": posterior.copy()
        })

        remaining_actions = [
            a for a in remaining_actions
            if a["name"] != action_name
        ]

        posterior = update_posterior_with_outcome(
            posterior,
            outcome_models[action_name][outcome]
        )

        trajectory[-1]["posterior_after"] = posterior.copy()

In [506]:
cost_only_results = []

for _, case in evaluation_cases.iterrows():

    result = evaluate_case_cost_only(
        case.to_dict()
    )

    cost_only_results.append(result)

print(
    f"Evaluated {len(cost_only_results)} cases"
)

Evaluated 40 cases


In [507]:
reports = [
    r for r in cost_only_results
    if r["decision"] == "REPORT"
]

escalations = [
    r for r in cost_only_results
    if r["decision"] == "ESCALATE"
]

correct = [
    r for r in reports
    if r["correct"]
]

print("Total cases:", len(cost_only_results))
print("Reports:", len(reports))
print("Escalations:", len(escalations))
print("Correct reports:", len(correct))
print("Accuracy:", len(correct) / len(cost_only_results))
print("Human-review rate:", len(escalations) / len(cost_only_results))

Total cases: 40
Reports: 25
Escalations: 15
Correct reports: 20
Accuracy: 0.5
Human-review rate: 0.375


In [508]:
results = []

for _, case in evaluation_cases.iterrows():

    result = evaluate_case(
        case.to_dict()
    )

    results.append(result)

print(f"Evaluated {len(results)} cases")

Evaluated 40 cases


In [509]:
reports = [
    r for r in results
    if r["decision"] == "REPORT"
]

escalations = [
    r for r in results
    if r["decision"] == "ESCALATE"
]

correct = [
    r for r in reports
    if r["correct"]
]

print("Total cases:", len(results))
print("Reports:", len(reports))
print("Escalations:", len(escalations))
print("Correct reports:", len(correct))
print("Accuracy among all cases:", len(correct) / len(results))
print("Human-review rate:", len(escalations) / len(results))

Total cases: 40
Reports: 26
Escalations: 14
Correct reports: 22
Accuracy among all cases: 0.55
Human-review rate: 0.35


In [510]:
top_posterior_correct = 0

for r in results:
    if not r["actions_taken"]:
        continue

    final_posterior = r["actions_taken"][-1]["posterior_after"]

    predicted_h = max(
        final_posterior,
        key=final_posterior.get
    )

    if predicted_h == r["true_root_cause"]:
        top_posterior_correct += 1

print(
    "Top-posterior accuracy:",
    top_posterior_correct / len(results)
)

Top-posterior accuracy: 0.8


In [511]:
BASELINE_ROOT_CAUSE = "Test"


def evaluate_baseline(evaluation_cases):
    results = []

    for _, case in evaluation_cases.iterrows():

        prediction = BASELINE_ROOT_CAUSE
        true_root_cause = case["true_root_cause"]

        results.append({
            "case_id": case["case_id"],
            "true_root_cause": true_root_cause,
            "final_prediction": prediction,
            "decision": "REPORT",
            "actions_taken": [],
            "total_cost": 0,
            "correct": prediction == true_root_cause
        })

    return results

baseline_results = evaluate_baseline(
    evaluation_cases
)

correct = [
    r for r in baseline_results
    if r["correct"]
]

print("Total cases:", len(baseline_results))
print("Correct:", len(correct))
print("Accuracy:", len(correct) / len(baseline_results))

Total cases: 40
Correct: 10
Accuracy: 0.25


In [512]:
from collections import Counter


def calculate_metrics(results, root_causes):
    total = len(results)

    reports = [
        r for r in results
        if r["decision"] == "REPORT"
    ]

    escalations = [
        r for r in results
        if r["decision"] == "ESCALATE"
    ]

    correct = [
        r for r in reports
        if r["correct"]
    ]

    # ---------------------------------------------
    # Basic metrics
    # ---------------------------------------------

    accuracy = len(correct) / total

    human_review_rate = len(escalations) / total

    report_precision = (
        len(correct) / len(reports)
        if reports else 0
    )

    average_cost = (
        sum(r["total_cost"] for r in results) / total
    )

    average_actions = (
        sum(len(r["actions_taken"]) for r in results) / total
    )

    # ---------------------------------------------
    # Confusion matrix
    # ---------------------------------------------

    confusion_matrix = {
        actual: {
            predicted: 0
            for predicted in root_causes
        }
        for actual in root_causes
    }

    for r in results:

        if r["decision"] != "REPORT":
            continue

        actual = r["true_root_cause"]
        predicted = r["final_prediction"]

        confusion_matrix[actual][predicted] += 1

    # ---------------------------------------------
    # Precision / Recall per root cause
    # ---------------------------------------------

    precision = {}
    recall = {}

    for h in root_causes:

        tp = confusion_matrix[h][h]

        predicted_positive = sum(
            confusion_matrix[actual][h]
            for actual in root_causes
        )

        actual_positive = sum(
            1
            for r in results
            if r["true_root_cause"] == h
        )

        precision[h] = (
            tp / predicted_positive
            if predicted_positive
            else 0
        )

        recall[h] = (
            tp / actual_positive
            if actual_positive
            else 0
        )

    macro_precision = sum(precision.values()) / len(precision)
    macro_recall = sum(recall.values()) / len(recall)

    return {
        "accuracy": accuracy,
        "human_review_rate": human_review_rate,
        "report_precision": report_precision,
        "average_cost": average_cost,
        "average_actions": average_actions,
        "confusion_matrix": confusion_matrix,
        "precision": precision,
        "recall": recall,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall
    }

In [513]:
root_causes = [
    "Code",
    "Test",
    "Dependency",
    "CI/Config"
]


eig_metrics = calculate_metrics(
    results,
    root_causes
)

cost_metrics = calculate_metrics(
    cost_only_results,
    root_causes
)

baseline_metrics = calculate_metrics(
    baseline_results,
    root_causes
)

In [514]:
print("EIG")
print(eig_metrics["precision"])
print(eig_metrics["recall"])

print("\nCOST")
print(cost_metrics["precision"])
print(cost_metrics["recall"])

print("\nBASELINE")
print(baseline_metrics["precision"])
print(baseline_metrics["recall"])

EIG
{'Code': 0, 'Test': 0.7, 'Dependency': 0.9090909090909091, 'CI/Config': 1.0}
{'Code': 0.0, 'Test': 0.7, 'Dependency': 1.0, 'CI/Config': 0.5}

COST
{'Code': 0, 'Test': 0.7, 'Dependency': 0.8888888888888888, 'CI/Config': 0.8333333333333334}
{'Code': 0.0, 'Test': 0.7, 'Dependency': 0.8, 'CI/Config': 0.5}

BASELINE
{'Code': 0, 'Test': 0.25, 'Dependency': 0, 'CI/Config': 0}
{'Code': 0.0, 'Test': 1.0, 'Dependency': 0.0, 'CI/Config': 0.0}


In [515]:
comparison = {
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Human-review rate",
        "Average cost",
        "Average actions",
        "Report precision"
    ],

    "EIG": [
        eig_metrics["accuracy"],
        eig_metrics["macro_precision"],
        eig_metrics["macro_recall"],
        eig_metrics["human_review_rate"],
        eig_metrics["average_cost"],
        eig_metrics["average_actions"],
        eig_metrics["report_precision"]
    ],

    "Cost-only": [
        cost_metrics["accuracy"],
        cost_metrics["macro_precision"],
        cost_metrics["macro_recall"],
        cost_metrics["human_review_rate"],
        cost_metrics["average_cost"],
        cost_metrics["average_actions"],
        cost_metrics["report_precision"]
    ],

    "Baseline": [
        baseline_metrics["accuracy"],
        baseline_metrics["macro_precision"],
        baseline_metrics["macro_recall"],
        baseline_metrics["human_review_rate"],
        baseline_metrics["average_cost"],
        baseline_metrics["average_actions"],
        baseline_metrics["report_precision"]
    ]
}

In [516]:
comparison_df = pd.DataFrame(comparison)

comparison_df

,Metric,EIG,Cost-only,Baseline
0,Accuracy,0.550000,0.500000,0.2500
1,Macro Precision,0.652273,0.605556,0.0625
2,Macro Recall,0.550000,0.500000,0.2500
3,Human-review rate,0.350000,0.375000,0.0000
4,Average cost,6.462500,6.925000,0.0000
5,Average actions,4.675000,5.225000,0.0000
6,Report precision,0.846154,0.800000,0.2500
